In [ ]:
!pip install praat-parselmouth opensmile librosa noisereduce joblib


In [ ]:
import numpy
import sklearn
import scipy

print(numpy.__version__)
print(sklearn.__version__)
print(scipy.__version__)


In [ ]:
import os
import opensmile
import numpy as np
import pandas as pd
from tqdm import tqdm


In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals
)


In [ ]:
copd_path = r"C:\Users\Seethalakshmi\Documents\COPD project\New COPD Patients Data"        # flat wav folder
normal_path = r"C:\Users\Seethalakshmi\Documents\COPD project\normal"    # normal_aa / ee / uu structure


In [ ]:
def extract_egemaps(file):
    try:
        features = smile.process_file(file)
        return features.values.flatten()
    except:
        return None


In [ ]:
copd_features = []
copd_files = [f for f in os.listdir(copd_path) if f.endswith(".wav")]

for f in tqdm(copd_files, desc="COPD Files"):
    full = os.path.join(copd_path, f)
    feat = extract_egemaps(full)
    if feat is not None:
        copd_features.append([f, "COPD"] + feat.tolist())


In [ ]:
normal_features = []

for vowel in os.listdir(normal_path):
    vowel_path = os.path.join(normal_path, vowel)

    if not os.path.isdir(vowel_path):
        continue

    for gender in os.listdir(vowel_path):
        gender_path = os.path.join(vowel_path, gender)

        if not os.path.isdir(gender_path):
            continue

        for loudness in os.listdir(gender_path):
            loud_path = os.path.join(gender_path, loudness)

            if not os.path.isdir(loud_path):
                continue

            wavs = [f for f in os.listdir(loud_path) if f.lower().endswith(".wav")]

            print(f"\n📂 Processing: {vowel} | {gender} | {loudness} → {len(wavs)} files")

            for f in tqdm(wavs, desc=f"{vowel}-{gender}-{loudness}"):
                full = os.path.join(loud_path, f)
                feat = extract_egemaps(full)
                if feat is not None:
                    normal_features.append([f, "Healthy"] + feat.tolist())


In [ ]:
len(normal_features)


In [ ]:
columns = ["filename", "label"] + list(smile.feature_names)

df_copd = pd.DataFrame(copd_features, columns=columns)
df_normal = pd.DataFrame(normal_features, columns=columns)

final_df = pd.concat([df_copd, df_normal], ignore_index=True)

final_df.head()


In [ ]:
final_df.to_csv("egemaps_newdataset.csv", index=False)
print("✔ Feature extraction complete and saved.")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("egemaps_newdataset.csv")

print(df.shape)
df.head()


In [ ]:
X = df.drop(columns=["filename", "label"])
y = df["label"].map({"Healthy": 0, "COPD": 1})

print(X.shape, y.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

rf_estimator = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rfe = RFE(
    estimator=rf_estimator,
    n_features_to_select=15
)

X_rfe = rfe.fit_transform(X_scaled, y)

rfe_features = X.columns[rfe.support_]

print("Top 15 RFE-selected features:")
print(list(rfe_features))


In [ ]:
X_rfe_df = pd.DataFrame(X_rfe, columns=rfe_features)
X_rfe_df["label"] = y.values

X_rfe_df.head()


In [ ]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    s1, s2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1)*s1 + (n2 - 1)*s2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

cohens_d_scores = {}

for feat in rfe_features:
    copd_vals = X_rfe_df[X_rfe_df["label"] == 1][feat]
    healthy_vals = X_rfe_df[X_rfe_df["label"] == 0][feat]
    cohens_d_scores[feat] = abs(cohens_d(copd_vals, healthy_vals))

cohens_d_df = (
    pd.DataFrame.from_dict(cohens_d_scores, orient="index", columns=["Cohens_d"])
    .sort_values(by="Cohens_d", ascending=False)
)

cohens_d_df


In [ ]:
top5_features = cohens_d_df.head(5).index.tolist()
print("Top 5 features based on Cohen's d:")
top5_features


In [ ]:
from scipy.stats import ttest_ind

t_test_results = []

for feat in top5_features:
    copd_vals = X_rfe_df[X_rfe_df["label"] == 1][feat]
    healthy_vals = X_rfe_df[X_rfe_df["label"] == 0][feat]
    
    t_stat, p_val = ttest_ind(copd_vals, healthy_vals, equal_var=False)
    
    t_test_results.append([feat, t_stat, p_val])

t_test_df = pd.DataFrame(
    t_test_results,
    columns=["Feature", "t_statistic", "p_value"]
)

t_test_df


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X_rfe, y, test_size=0.2, random_state=42, stratify=y
)

svm = SVC(kernel="rbf", probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)

svm.fit(X_train, y_train)
rf.fit(X_train, y_train)

svm_preds = svm.predict(X_test)
rf_preds = rf.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, svm_preds))
print("RF Accuracy:", accuracy_score(y_test, rf_preds))

print("SVM AUC:", roc_auc_score(y_test, svm.predict_proba(X_test)[:,1]))
print("RF AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))


In [ ]:
groups = df["filename"].apply(lambda x: x.split("_")[0])


In [ ]:
groups.unique()


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score
import numpy as np

svm_scores = []
rf_scores = []

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X_rfe, y, groups):
    X_train, X_test = X_rfe[train_idx], X_rfe[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    svm.fit(X_train, y_train)
    rf.fit(X_train, y_train)

    svm_scores.append(accuracy_score(y_test, svm.predict(X_test)))
    rf_scores.append(accuracy_score(y_test, rf.predict(X_test)))

print("SVM LOSO Accuracy:", np.mean(svm_scores))
print("RF LOSO Accuracy:", np.mean(rf_scores))


In [ ]:
X_all = final_df.drop(columns=["filename", "label"])
X_all = X_all.apply(pd.to_numeric, errors="coerce")
X_all = X_all.fillna(X_all.mean())

y = final_df["label"].map({"Healthy": 0, "COPD": 1}).values


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (important for SVM)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Models
svm = SVC(kernel="rbf", probability=True, random_state=42)
rf  = RandomForestClassifier(n_estimators=200, random_state=42)

# Train
svm.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Predict
svm_pred = svm.predict(X_test)
rf_pred  = rf.predict(X_test)

# Results
print("===== Entire eGeMAPS (All Features) =====")
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("RF Accuracy :", accuracy_score(y_test, rf_pred))
print("SVM AUC:", roc_auc_score(y_test, svm.predict_proba(X_test)[:,1]))
print("RF AUC :", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))


In [ ]:
top5 = cohens_d_df.head(5).index.tolist()
print("Top 5 Clinically Relevant Features:\n", top5)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for feature in top5:
    plt.figure(figsize=(4,4))
    sns.boxplot(x="label", y=feature, data=df_rfe)
    plt.title(f"Clinical Separation: {feature}")
    plt.tight_layout()
    plt.show()


In [ ]:
def signif(p):
    if p < 0.001:
        return "*** (Highly Significant)"
    elif p < 0.01:
        return "** (Very Significant)"
    elif p < 0.05:
        return "* (Significant)"
    else:
        return "Not Significant"

stats_df["Significance"] = stats_df["p_value"].apply(signif)
stats_df


In [ ]:
X_top5 = final_df[top5]
y_top5 = final_df["label"].map({"Healthy":0, "COPD":1})


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler5 = StandardScaler()
X_top5_scaled = scaler5.fit_transform(X_top5)


In [ ]:
from sklearn.model_selection import train_test_split

X_train5, X_test5, y_train5, y_test5 = train_test_split(
    X_top5_scaled, y_top5, test_size=0.2, random_state=42, stratify=y_top5)


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score

svm5 = SVC(kernel="rbf", probability=True)
svm5.fit(X_train5, y_train5)

pred5 = svm5.predict(X_test5)
auc5 = roc_auc_score(y_test5, svm5.predict_proba(X_test5)[:,1])

print("Top5 SVM Accuracy:", accuracy_score(y_test5, pred5))
print("Top5 SVM AUC:", auc5)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf5 = RandomForestClassifier(n_estimators=200, random_state=42)
rf5.fit(X_train5, y_train5)

rf_pred5 = rf5.predict(X_test5)
rf_auc5 = roc_auc_score(y_test5, rf5.predict_proba(X_test5)[:,1])

print("Top5 RF Accuracy:", accuracy_score(y_test5, rf_pred5))
print("Top5 RF AUC:", rf_auc5)


In [ ]:
print("\nClinical Insight:")
print("The top 5 features derived using RFE + Cohen’s d alone achieve strong classification performance,")
print("indicating they serve as reliable acoustic biomarkers for COPD detection even without full feature set.")


In [ ]:
final_df["speaker"] = final_df["filename"].apply(lambda x: x.split("_")[0])


In [ ]:
X5 = final_df[top5]
y5 = final_df["label"].map({"Healthy":0, "COPD":1})
groups = final_df["speaker"]


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(X5, y5, groups):
    X_train5, X_test5 = X5.iloc[train_idx], X5.iloc[test_idx]
    y_train5, y_test5 = y5.iloc[train_idx], y5.iloc[test_idx]

scaler = StandardScaler()
X_train5 = scaler.fit_transform(X_train5)
X_test5 = scaler.transform(X_test5)


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# SVM
svm5 = SVC(kernel="rbf", probability=True)
svm5.fit(X_train5, y_train5)

svm_pred5 = svm5.predict(X_test5)
svm_auc5 = roc_auc_score(y_test5, svm5.predict_proba(X_test5)[:,1])

print("Speaker-independent SVM Accuracy:", accuracy_score(y_test5, svm_pred5))
print("Speaker-independent SVM AUC:", svm_auc5)

# RF
rf5 = RandomForestClassifier(n_estimators=200, random_state=42)
rf5.fit(X_train5, y_train5)

rf_pred5 = rf5.predict(X_test5)
rf_auc5 = roc_auc_score(y_test5, rf5.predict_proba(X_test5)[:,1])

print("Speaker-independent RF Accuracy:", accuracy_score(y_test5, rf_pred5))
print("Speaker-independent RF AUC:", rf_auc5)


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X5 = final_df[top5]
y5 = final_df["label"].map({"Healthy":0, "COPD":1})
groups = final_df["speaker"]


In [ ]:
logo = LeaveOneGroupOut()

svm_acc = []

for train_idx, test_idx in logo.split(X5, y5, groups):
    X_train, X_test = X5.iloc[train_idx], X5.iloc[test_idx]
    y_train, y_test = y5.iloc[train_idx], y5.iloc[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = SVC(kernel="rbf")
    model.fit(X_train, y_train)

    svm_acc.append(model.score(X_test, y_test))

print("LOSO SVM Accuracy:", np.mean(svm_acc))


In [ ]:
rf_acc = []

for train_idx, test_idx in logo.split(X5, y5, groups):
    X_train, X_test = X5.iloc[train_idx], X5.iloc[test_idx]
    y_train, y_test = y5.iloc[train_idx], y5.iloc[test_idx]

    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    rf_acc.append(model.score(X_test, y_test))

print("LOSO RF Accuracy:", np.mean(rf_acc))


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, roc_curve, auc
import numpy as np

X15 = final_df[selected_features]
y15 = final_df["label"].map({"Healthy":0, "COPD":1})
groups = final_df["speaker"]

logo = LeaveOneGroupOut()

y_true_15, y_pred_15, y_score_15 = [], [], []

for train_idx, test_idx in logo.split(X15, y15, groups):
    X_train, X_test = X15.iloc[train_idx], X15.iloc[test_idx]
    y_train, y_test = y15.iloc[train_idx], y15.iloc[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = SVC(kernel="rbf", probability=True)
    model.fit(X_train, y_train)

    y_true_15.extend(y_test)
    y_pred_15.extend(model.predict(X_test))
    y_score_15.extend(model.predict_proba(X_test)[:,1])


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm15 = confusion_matrix(y_true_15, y_pred_15)

plt.figure()
plt.imshow(cm15)
plt.title("Confusion Matrix - Top 15 (LOSO)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr15, tpr15, _ = roc_curve(y_true_15, y_score_15)
roc_auc15 = auc(fpr15, tpr15)

plt.figure()
plt.plot(fpr15, tpr15, label=f"AUC = {roc_auc15:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.legend()
plt.title("ROC - Top 15 Features (LOSO)")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.tight_layout()
plt.show()


In [ ]:
X5 = final_df[top5]
y5 = final_df["label"].map({"Healthy":0, "COPD":1})

y_true_5, y_pred_5, y_score_5 = [], [], []

for train_idx, test_idx in logo.split(X5, y5, groups):
    X_train, X_test = X5.iloc[train_idx], X5.iloc[test_idx]
    y_train, y_test = y5.iloc[train_idx], y5.iloc[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = SVC(kernel="rbf", probability=True)
    model.fit(X_train, y_train)

    y_true_5.extend(y_test)
    y_pred_5.extend(model.predict(X_test))
    y_score_5.extend(model.predict_proba(X_test)[:,1])


In [ ]:
cm5 = confusion_matrix(y_true_5, y_pred_5)

plt.figure()
plt.imshow(cm5)
plt.title("Confusion Matrix - Top 5 (LOSO)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.tight_layout()
plt.show()

fpr5, tpr5, _ = roc_curve(y_true_5, y_score_5)
roc_auc5 = auc(fpr5, tpr5)

plt.figure()
plt.plot(fpr5, tpr5, label=f"AUC = {roc_auc5:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.legend()
plt.title("ROC - Top 5 Features (LOSO)")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score

acc15 = accuracy_score(y_true_15, y_pred_15)
print("LOSO Accuracy - Top 15 Features:", acc15)


In [ ]:
acc5 = accuracy_score(y_true_5, y_pred_5)
print("LOSO Accuracy - Top 5 Features:", acc5)


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

def run_loso(df, feature_list, title=""):
    
    X = df[feature_list].values
    y = df["label"].map({"Healthy":0, "COPD":1}).values
    groups = df["subject"].values
    
    logo = LeaveOneGroupOut()
    
    y_true_all = []
    y_pred_svm = []
    y_score_svm = []
    y_pred_rf = []
    y_score_rf = []
    
    for train_idx, test_idx in logo.split(X, y, groups):
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        svm = SVC(kernel="rbf", probability=True)
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        
        svm.fit(X_train, y_train)
        rf.fit(X_train, y_train)
        
        y_true_all.extend(y_test)
        y_pred_svm.extend(svm.predict(X_test))
        y_score_svm.extend(svm.predict_proba(X_test)[:,1])
        
        y_pred_rf.extend(rf.predict(X_test))
        y_score_rf.extend(rf.predict_proba(X_test)[:,1])
    
    acc_svm = accuracy_score(y_true_all, y_pred_svm)
    auc_svm = roc_auc_score(y_true_all, y_score_svm)
    
    acc_rf = accuracy_score(y_true_all, y_pred_rf)
    auc_rf = roc_auc_score(y_true_all, y_score_rf)
    
    print(f"\n===== LOSO Results: {title} =====")
    print("SVM Accuracy :", round(acc_svm,3))
    print("SVM AUC      :", round(auc_svm,3))
    print("RF Accuracy  :", round(acc_rf,3))
    print("RF AUC       :", round(auc_rf,3))
    
    return y_true_all, y_score_svm, y_score_rf


In [ ]:
X = final_df.drop(columns=["filename","label"])
y = final_df["label"].map({"Healthy":0,"COPD":1})


In [ ]:
import numpy as np

# keep only numeric feature columns
numeric_df = final_df.select_dtypes(include=[np.number])

# add label + subject back
numeric_df["label"] = final_df["label"].values


In [ ]:
X = numeric_df.drop(columns=["label"])
y = numeric_df["label"].map({"Healthy":0,"COPD":1})


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.svm import SVC

svm = SVC(kernel="linear")
rfe = RFE(svm, n_features_to_select=15)
rfe.fit(X, y)

top15 = X.columns[rfe.support_].tolist()
print(top15)


In [ ]:
import numpy as np

copd = final_df[final_df["label"]=="COPD"][top15]
healthy = final_df[final_df["label"]=="Healthy"][top15]

d_vals = {}

for f in top15:
    m1, m2 = copd[f].mean(), healthy[f].mean()
    s1, s2 = copd[f].std(), healthy[f].std()
    pooled = np.sqrt((s1**2 + s2**2)/2)
    d_vals[f] = abs((m1-m2)/pooled)

top5 = sorted(d_vals, key=d_vals.get, reverse=True)[:5]
print("Top 5 features:\n", top5)


In [ ]:
final_df["group"] = final_df["filename"].str.extract(r'(CP\d+)')


In [ ]:
# Replace NaN with column mean (best practice for medical features)
final_df[top15] = final_df[top15].fillna(final_df[top15].mean())
final_df[top5]  = final_df[top5].fillna(final_df[top5].mean())


In [ ]:
vowels = ["AA","EE","UU"]

for v in vowels:
    
    df_v = final_df[final_df["filename"].str.contains(v)]
    
    print("\n-------------------------")
    print("Vowel:", v)
    print("Samples:", len(df_v))


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

vowels = ["AA","EE","UU"]
gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

for v in vowels:
    
    df_v = final_df[final_df["filename"].str.contains(v)].copy()
    df_v = df_v.dropna(subset=["group"])
    
    X = df_v[top15].apply(pd.to_numeric, errors="coerce")
    y = df_v["label"].map({"Healthy":0,"COPD":1})
    groups = df_v["group"]
    
    X = X.dropna(axis=1, how="all")
    
    print("\n-------------------------")
    print("Vowel:", v)
    print("Samples:", len(df_v))
    print("Class counts:\n", y.value_counts())
    
    # 🚨 Skip if only one class present
    if y.nunique() < 2:
        print("Skipped — only one class present for this vowel.")
        continue
    
    svm_acc, rf_acc = [], []
    
    for train_idx, test_idx in gss.split(X, y, groups):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        imp = SimpleImputer(strategy="mean")
        X_train = imp.fit_transform(X_train)
        X_test  = imp.transform(X_test)
        
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test  = sc.transform(X_test)
        
        svm = SVC(kernel="rbf", probability=True)
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        
        svm.fit(X_train, y_train)
        rf.fit(X_train, y_train)
        
        svm_acc.append(accuracy_score(y_test, svm.predict(X_test)))
        rf_acc.append(accuracy_score(y_test, rf.predict(X_test)))
    
    print("Mean SVM Accuracy:", round(np.mean(svm_acc),3))
    print("Mean RF Accuracy :", round(np.mean(rf_acc),3))


In [ ]:
print(final_df.columns)


In [ ]:
import os
import pandas as pd

normal_root = r"C:\Users\Seethalakshmi\Documents\COPD project\normal"
copd_root  = r"C:\Users\Seethalakshmi\Documents\COPD project\New COPD Patients Data"

rows = []

# -------- NORMAL DATA (metadata from folders) --------
for root, dirs, files in os.walk(normal_root):
    for file in files:
        if file.endswith(".wav"):
            
            full = os.path.join(root, file)
            parts = full.lower().split(os.sep)
            
            try:
                gender = "female" if "female" in parts else "male"
                
                if "soft" in parts:
                    loudness = "soft"
                elif "comfortable" in parts:
                    loudness = "comfortable"
                elif "loud" in parts:
                    loudness = "loud"
                else:
                    loudness = None
                
                name = file.replace(".wav","")
                spk, vowel, lvl = name.split("_")
                
                rows.append([file,"Healthy",spk,vowel,loudness,gender])
            except:
                continue


# -------- CREATE DATAFRAME --------
meta_df = pd.DataFrame(rows,columns=[
    "filename","label","speaker","vowel","loudness","gender"
])

print(meta_df.head())
print("\nCounts by loudness:\n", meta_df["loudness"].value_counts())


In [ ]:
# -------- COPD DATA (recursive walk) --------
for root, dirs, files in os.walk(copd_root):
    for file in files:
        if file.endswith(".wav"):
            
            name = file.replace(".wav","").replace("Copy of ","")
            
            try:
                spk, vowel, lvl, cond = name.split("_")
                
                if lvl == "S":
                    loudness = "soft"
                elif lvl == "C":
                    loudness = "comfortable"
                elif lvl == "L":
                    loudness = "loud"
                else:
                    loudness = None
                
                gender = None  # COPD dataset has no gender folders
                
                rows.append([file,"COPD",spk,vowel,loudness,gender])
            except:
                continue
meta_df = pd.DataFrame(rows,columns=[
    "filename","label","speaker","vowel","loudness","gender"
])

print(meta_df.tail())
print("\nLabel counts:\n", meta_df["label"].value_counts())


In [ ]:
final_df = final_df.merge(meta_df, on=["filename","label"], how="left")

print(final_df.columns)   # confirm loudness & gender are now present


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer
import numpy as np

gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# ---------------- LOUDNESS ANALYSIS ----------------
print("\n=========== Loudness-wise COPD vs Healthy ===========")

for lvl in ["soft","comfortable","loud"]:
    
    df_l = final_df[final_df["loudness"] == lvl].copy()
    df_l = df_l.dropna(subset=["group"])
    
    X = df_l[top15].apply(pd.to_numeric, errors="coerce")
    y = df_l["label"].map({"Healthy":0,"COPD":1})
    groups = df_l["group"]
    
    print(f"\nLoudness: {lvl}")
    print("Samples:", len(df_l))
    print("Class counts:\n", y.value_counts())
    
    if y.nunique() < 2:
        print("Skipped — only one class present.")
        continue
    
    svm_acc, rf_acc, svm_auc, rf_auc = [], [], [], []
    
    for train_idx, test_idx in gss.split(X, y, groups):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        imp = SimpleImputer(strategy="mean")
        X_train = imp.fit_transform(X_train)
        X_test  = imp.transform(X_test)
        
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test  = sc.transform(X_test)
        
        svm = SVC(kernel="rbf", probability=True)
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        
        svm.fit(X_train, y_train)
        rf.fit(X_train, y_train)
        
        svm_acc.append(accuracy_score(y_test, svm.predict(X_test)))
        rf_acc.append(accuracy_score(y_test, rf.predict(X_test)))
        
        svm_auc.append(roc_auc_score(y_test, svm.predict_proba(X_test)[:,1]))
        rf_auc.append(roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))
    
    print("Mean SVM Accuracy:", round(np.mean(svm_acc),3))
    print("Mean RF Accuracy :", round(np.mean(rf_acc),3))
    print("Mean SVM AUC     :", round(np.mean(svm_auc),3))
    print("Mean RF AUC      :", round(np.mean(rf_auc),3))


# ---------------- GENDER FEATURE DISTRIBUTION ----------------
print("\n=========== Gender-wise Feature Behavior (Healthy only) ===========")

df_gender = final_df[final_df["label"]=="Healthy"]

print(df_gender["gender"].value_counts())


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

# COPD only
copd_df = final_df[final_df["label"]=="COPD"].copy()

print("COPD loudness counts:\n", copd_df["loudness"].value_counts())

# --- ANOVA per feature ---
anova_results = {}

for feat in top15:
    
    soft = copd_df[copd_df["loudness"]=="soft"][feat].dropna()
    comf = copd_df[copd_df["loudness"]=="comfortable"][feat].dropna()
    loud = copd_df[copd_df["loudness"]=="loud"][feat].dropna()
    
    if len(soft)>2 and len(comf)>2 and len(loud)>2:
        f,p = f_oneway(soft,comf,loud)
        anova_results[feat] = p

anova_df = pd.DataFrame.from_dict(anova_results,orient="index",columns=["p_value"])
anova_df = anova_df.sort_values("p_value")

print("\nSignificant loudness-sensitive features (COPD only):")
print(anova_df.head(10))

# --- Boxplots for top significant features ---
top_feats = anova_df.head(5).index

for feat in top_feats:
    plt.figure(figsize=(7,5))
    sns.boxplot(x="loudness", y=feat, data=copd_df)
    plt.title(f"{feat} variation across loudness (COPD)")
    plt.show()


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

vowels = final_df["vowel"].dropna().unique()   # remove NaN vowels first

print("\n=========== Cross Vowel Generalization (SVM-RBF) ===========")

for train_v in vowels:
    for test_v in vowels:
        if train_v != test_v:

            train_df = final_df[final_df["vowel"] == train_v]
            test_df  = final_df[final_df["vowel"] == test_v]

            # 🚨 Skip if either split is empty
            if len(train_df) == 0:
                print(f"Train {train_v} → Test {test_v} : SKIPPED (no train samples)")
                continue

            if len(test_df) == 0:
                print(f"Train {train_v} → Test {test_v} : SKIPPED (no test samples)")
                continue

            # 🚨 Skip if train has only one class
            if train_df["label"].nunique() < 2:
                print(f"Train {train_v} → Test {test_v} : SKIPPED (single class in train)")
                continue

            X_train = train_df[top15].astype(float)
            y_train = train_df["label"].map({"Healthy":0, "COPD":1})

            X_test = test_df[top15].astype(float)
            y_test = test_df["label"].map({"Healthy":0, "COPD":1})

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            clf = SVC(kernel="rbf", C=10, gamma="scale")
            clf.fit(X_train, y_train)

            preds = clf.predict(X_test)
            acc = accuracy_score(y_test, preds)

            print(f"Train {train_v} → Test {test_v} : {acc:.3f}")


In [ ]:
!pip install shap


In [ ]:
import shap
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# ---------- Prepare numeric feature matrix ----------
X_shap = final_df[top15].apply(pd.to_numeric, errors="coerce")
X_shap = X_shap.fillna(X_shap.mean())

y_shap = final_df["label"].map({"Healthy":0, "COPD":1})

# ---------- Train SVM ----------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_shap)

model = SVC(kernel="rbf", C=10, gamma="scale", probability=True)
model.fit(X_scaled, y_shap)

# ---------- SHAP background & test ----------
background = pd.DataFrame(X_scaled[:40], columns=top15)
test_data  = pd.DataFrame(X_scaled[40:60], columns=top15)

explainer = shap.KernelExplainer(model.predict_proba, background)

shap_values = explainer.shap_values(test_data)

# ---------- Ensure correct dimensions ----------
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

if sv.shape[1] != test_data.shape[1]:
    sv = sv[:, :test_data.shape[1]]

# ---------- Plot ----------
shap.summary_plot(sv, test_data, feature_names=top15)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------- Step 1: Filter COPD -------------------
copd_df = final_df[final_df["label"]=="COPD"].copy()

# ------------------- Step 2: Create a severity proxy -------------------
# Option 1: Mean of top features
copd_df["severity_proxy"] = copd_df[top15].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Option 2: (Optional) normalize to 0-1 scale for better plotting
copd_df["severity_proxy_norm"] = (copd_df["severity_proxy"] - copd_df["severity_proxy"].min()) / \
                                 (copd_df["severity_proxy"].max() - copd_df["severity_proxy"].min())

# ------------------- Step 3: (Optional) categorize into Mild/Moderate/Severe -------------------
# Using tertiles
copd_df["severity_plot"] = pd.qcut(copd_df["severity_proxy_norm"], q=3, labels=["Mild","Moderate","Severe"])

# ------------------- Step 4: Plot trends of top features -------------------
plt.figure(figsize=(12,6))
for feat in top15[:5]:  # top 5 features for clarity
    sns.scatterplot(x=copd_df["severity_proxy_norm"], y=copd_df[feat], label=feat)

plt.xlabel("Severity Proxy (normalized)")
plt.ylabel("Feature Value")
plt.title("Feature Trend across COPD Severity Proxy")
plt.legend()
plt.show()

# ------------------- Step 5: Boxplot by severity category -------------------
plt.figure(figsize=(12,6))
for i, feat in enumerate(top15[:5], start=1):
    plt.subplot(1,5,i)
    sns.boxplot(x="severity_plot", y=feat, data=copd_df)
    plt.title(feat)
    plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# ------------------- Step 6: Optional: Correlation with proxy -------------------
corrs = copd_df[top15].corrwith(copd_df["severity_proxy"])
print("Correlation of features with severity proxy:")
print(corrs.sort_values(ascending=False))


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# ---------- Prepare data ----------
X = final_df[top15].apply(pd.to_numeric, errors="coerce").fillna(final_df[top15].mean())
y = final_df["label"].map({"Healthy":0,"COPD":1})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

noise_levels = [0.01, 0.05, 0.1, 0.2]

print("\n=========== Noise Robustness Test ===========")

for nl in noise_levels:
    noise = np.random.normal(0, nl, X_scaled.shape)
    X_noisy = X_scaled + noise
    
    # ----- SVM -----
    svm_clf = SVC(kernel="rbf", C=10, gamma="scale")
    svm_clf.fit(X_scaled, y)  # train on clean data
    svm_preds = svm_clf.predict(X_noisy)
    svm_acc = accuracy_score(y, svm_preds)
    
    # ----- Random Forest -----
    rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_clf.fit(X_scaled, y)   # train on clean data
    rf_preds = rf_clf.predict(X_noisy)
    rf_acc = accuracy_score(y, rf_preds)
    
    print(f"Noise {nl*100:.0f}% → SVM Acc: {svm_acc:.3f} | RF Acc: {rf_acc:.3f}")


In [ ]:
import matplotlib.pyplot as plt

# ---------- Prepare data ----------
X = final_df[top15].apply(pd.to_numeric, errors="coerce").fillna(final_df[top15].mean())
y = final_df["label"].map({"Healthy":0,"COPD":1})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

noise_levels = [0.01, 0.05, 0.1, 0.2]

svm_accuracies = []
rf_accuracies = []

for nl in noise_levels:
    noise = np.random.normal(0, nl, X_scaled.shape)
    X_noisy = X_scaled + noise
    
    # ----- SVM -----
    svm_clf = SVC(kernel="rbf", C=10, gamma="scale")
    svm_clf.fit(X_scaled, y)
    svm_preds = svm_clf.predict(X_noisy)
    svm_accuracies.append(accuracy_score(y, svm_preds))
    
    # ----- Random Forest -----
    rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_clf.fit(X_scaled, y)
    rf_preds = rf_clf.predict(X_noisy)
    rf_accuracies.append(accuracy_score(y, rf_preds))

# ---------- Plotting ----------
plt.figure(figsize=(8,5))
plt.plot([nl*100 for nl in noise_levels], svm_accuracies, marker='o', label='SVM')
plt.plot([nl*100 for nl in noise_levels], rf_accuracies, marker='s', label='Random Forest')
plt.xlabel("Noise Level (%)")
plt.ylabel("Accuracy")
plt.title("Noise Robustness Test: SVM vs Random Forest")
plt.xticks([nl*100 for nl in noise_levels])
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()


In [ ]:
final_df["group"].isna().sum()


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ------------------------------------
# Prepare data
# ------------------------------------
X = final_df[top15].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean())

y = final_df["label"].map({"Healthy": 0, "COPD": 1}).values
groups = final_df["speaker_id"].values

logo = LeaveOneGroupOut()

svm_preds_all = []
rf_preds_all = []
y_true_all = []

# ------------------------------------
# LOSO loop
# ------------------------------------
for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ---- SVM ----
    svm = SVC(kernel="rbf", C=10, gamma="scale")
    svm.fit(X_train, y_train)
    svm_pred = svm.predict(X_test)

    # ---- RF ----
    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    # Store predictions
    svm_preds_all.extend(svm_pred)
    rf_preds_all.extend(rf_pred)
    y_true_all.extend(y_test)

# ------------------------------------
# Final LOSO accuracy
# ------------------------------------
svm_loso_acc = accuracy_score(y_true_all, svm_preds_all)
rf_loso_acc  = accuracy_score(y_true_all, rf_preds_all)

print("\n=========== Speaker-Independent LOSO Results ===========")
print(f"SVM LOSO Accuracy : {svm_loso_acc:.3f}")
print(f"RF  LOSO Accuracy : {rf_loso_acc:.3f}")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Confusion matrices
cm_svm = confusion_matrix(y_true_all, svm_preds_all)
cm_rf  = confusion_matrix(y_true_all, rf_preds_all)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay(
    confusion_matrix=cm_svm,
    display_labels=["Healthy", "COPD"]
).plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title("SVM – LOSO Confusion Matrix")

ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["Healthy", "COPD"]
).plot(ax=axes[1], cmap="Greens", values_format="d")
axes[1].set_title("Random Forest – LOSO Confusion Matrix")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

svm_probs_all = []
rf_probs_all = []
y_true_all = []

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ---- SVM ----
    svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True)
    svm.fit(X_train, y_train)
    svm_prob = svm.predict_proba(X_test)[:, 1]

    # ---- RF ----
    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)
    rf_prob = rf.predict_proba(X_test)[:, 1]

    svm_probs_all.extend(svm_prob)
    rf_probs_all.extend(rf_prob)
    y_true_all.extend(y_test)

# Convert to numpy
y_true_all = np.array(y_true_all)
svm_probs_all = np.array(svm_probs_all)
rf_probs_all = np.array(rf_probs_all)


In [ ]:
# ROC for SVM
fpr_svm, tpr_svm, _ = roc_curve(y_true_all, svm_probs_all)
auc_svm = auc(fpr_svm, tpr_svm)

# ROC for RF
fpr_rf, tpr_rf, _ = roc_curve(y_true_all, rf_probs_all)
auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"RF (AUC = {auc_rf:.3f})")
plt.plot([0, 1], [0, 1], "--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Speaker-Independent LOSO ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()
